# detach-clone-snapshot — worked example 1: Record a parameter trajectory across SGD steps without aliasing

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-clone-snapshot`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When you run an optimizer loop and want to keep the value of a parameter at every step, appending the live tensor itself stores an *alias*. Because the optimizer updates the parameter **in place**, every entry in the list ends up pointing at the same storage and shows the final value. `p.detach().clone()` records an inert, independent copy: `detach()` severs it from the autograd graph and `clone()` gives it its own storage so later in-place updates can't change it.

## Worked solution

**Goal:** fit `p` so it approaches a target via plain gradient descent, and keep a faithful per-step record of `p`.

1. **Make a leaf parameter.** `p = t.tensor([0.0], requires_grad=True)` is the live tensor the optimizer mutates in place each step.
2. **Loss + backward.** `loss = (p - target)**2` builds a graph; `loss.backward()` fills `p.grad`.
3. **In-place update.** Under `t.no_grad()`, `p -= lr * p.grad` mutates `p`'s storage directly. This is exactly why naive snapshotting fails: the storage is reused.
4. **Snapshot correctly.** `traj.append(p.detach().clone())`. `detach()` returns a view that shares storage but is graph-free; `clone()` then allocates *fresh* storage holding the current values. The appended tensor is frozen at this step's value.
5. **Why detach THEN clone.** Appending `p` alone would alias live storage (all entries equal the final value). Appending `p.detach()` alone is graph-free but STILL shares storage, so it would also track the in-place writes. Only the `.clone()` breaks the storage link.
6. **Verify.** Stacking the trajectory and checking that early entries differ from the last confirms no aliasing occurred.

In [ ]:
t.manual_seed(0)

def snapshot_trajectory(target=3.0, lr=0.1, steps=5):
    p = t.tensor([0.0], requires_grad=True)
    traj = []
    for _ in range(steps):
        loss = (p - target) ** 2
        loss.backward()
        with t.no_grad():
            p -= lr * p.grad
            p.grad.zero_()
        traj.append(p.detach().clone())
    return t.stack(traj)

traj = snapshot_trajectory()
print("trajectory values:", traj.squeeze().tolist())
print("first != last (no aliasing):", bool(traj[0] != traj[-1]))
print("any requires_grad:", any(x.requires_grad for x in traj))